<a href="https://colab.research.google.com/github/vsolanki76771215/Student_MLE_MiniProject_EDA/blob/main/ExperimentWithVariousModels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ============================================================
# STEP 1: Import Required Libraries
# ============================================================

import os
import time
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_validate, GridSearchCV

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier,
    VotingClassifier,
    StackingClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

In [7]:
# ============================================================
# STEP 2: Load Dataset from Colab sample_data Folder
# ============================================================

# In this project, the processed machine learning dataset has already
# been uploaded to the Colab environment and stored in the sample_data
# directory. We will read the CSV directly from that location.

# Specify the dataset location inside sample_data.
# Replace the filename if your uploaded file has a different name.

DATA_PATH = "/content/sample_data/all_patches_features_labels_s2_ndvi.csv"

# Verify that the file exists before loading.

if os.path.exists(DATA_PATH):
    print("Dataset found.")
else:
    print("Dataset not found. Check filename and location.")

df = pd.read_csv(DATA_PATH)

# Display basic information about the dataset.

print("Dataset Shape:", df.shape)

# Preview first few records.

df.head()


Dataset found.
Dataset Shape: (91454, 8)


,aoi,patch_file,ndvi_2018_mean,ndvi_2022_mean,ndvi_diff,loss_fraction,loss_binary,treecover_mean
0,la_pampa,patch_000000.npz,NaN,0.734583,NaN,0.000000,0,99.921875
1,la_pampa,patch_000001.npz,NaN,0.330006,NaN,0.302734,1,99.710938
2,la_pampa,patch_000002.npz,NaN,0.951192,NaN,0.314453,1,97.947266
3,la_pampa,patch_000003.npz,NaN,0.955078,NaN,0.034180,0,99.526367
4,la_pampa,patch_000004.npz,NaN,0.986328,NaN,0.000000,0,99.088867


In [8]:
# ============================================================
# STEP 3: Create Output Directory for Model Results
# ============================================================

# All experiment outputs, tuning results, trained models,
# and evaluation summaries will be saved in this folder.

OUTPUT_DIR = "/content/model_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Output Directory:", OUTPUT_DIR)

Output Directory: /content/model_results


In [9]:
# ============================================================
# STEP 4: Inspect Dataset
# ============================================================

print("Columns:")
print(df.columns)

print("\nMissing values:")
print(df.isna().sum())

print("\nClass balance:")
print(df["loss_binary"].value_counts())
print(df["loss_binary"].value_counts(normalize=True))

Columns:
Index(['aoi', 'patch_file', 'ndvi_2018_mean', 'ndvi_2022_mean', 'ndvi_diff',
       'loss_fraction', 'loss_binary', 'treecover_mean'],
      dtype='object')

Missing values:
aoi                   0
patch_file            0
ndvi_2018_mean    23173
ndvi_2022_mean        0
ndvi_diff         23173
loss_fraction         0
loss_binary           0
treecover_mean        0
dtype: int64

Class balance:
loss_binary
0    79493
1    11961
Name: count, dtype: int64
loss_binary
0    0.869213
1    0.130787
Name: proportion, dtype: float64


In [10]:
# ============================================================
# STEP 5: Define Features and Target
# ============================================================

# Features are NDVI-based satellite measurements.
# Target is binary deforestation label:
# 0 = no major forest loss
# 1 = forest loss detected

feature_cols = [
    "ndvi_2018_mean",
    "ndvi_2022_mean",
    "ndvi_diff"
]

target_col = "loss_binary"

df_model = df.dropna(subset=feature_cols + [target_col])

X = df_model[feature_cols]
y = df_model[target_col].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (68281, 3)
y shape: (68281,)


In [11]:
# ============================================================
# STEP 6: Train/Test Split
# ============================================================

# The train/test split gives us a final holdout test set.
# Cross-validation will be applied only on the training set.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (54624, 3)
Testing set: (13657, 3)


In [12]:
# ============================================================
# STEP 8: Define Performance Metrics
# ============================================================

# F1 is the main metric because deforestation detection may have class imbalance.
# Accuracy alone can be misleading when most patches are non-deforestation.

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}